In [1]:
import json
from transformers import AutoTokenizer
import os
from tqdm import tqdm

In [5]:
!pip install transformers
!pip install tqdm

In [2]:
HYPERP_FNAME = 'hyperp.json'
GENINFO_FNAME = 'generation_info.json'
NUM_TOKENS_FIELD = 'num_tokens'
BASE_MODELS_DIR = '/home/jovyan/work/RAG-project-SMILES-2024-/models'

DATASETS_CONFIG = {
    #'squadv2': ['logs_v2', 'logs'],
    'mtssquad': ['logs_v2'],
    #'mlqa/de': ['logs'],
    #'mlqa/hi': ['logs'],
    #'mlqa/vi': ['logs']
}
    

In [3]:
def load_json(json_path: str):
    with open(json_path, 'r', encoding='utf-8') as fd:
        output = json.loads(fd.read())
    return output

def save_json(json_path: str, data: object):
    with open(json_path, 'w', encoding='utf-8') as fd:
        fd.write(json.dumps(data))

In [4]:
for ds_path, logs_dir in DATASETS_CONFIG.items():
    for logdir in logs_dir:
        print(ds_path, logdir)
        logpath = f"{ds_path}/{logdir}"
        exps_dir = os.listdir(logpath)
        exps_dir = list(filter(lambda path: path.startswith("v"), exps_dir))

        for expdir in exps_dir:
            cur_exppath = f"{logpath}/{expdir}"
            print(cur_exppath)
            
            geninfo_path = f"{cur_exppath}/{GENINFO_FNAME}"
            generation_info = load_json(geninfo_path)

            # validating
            if not 'metainfo' in generation_info[0]:
                print(f"not valid: {geninfo_path}")
                continue

            hyperp_path = f"{cur_exppath}/{HYPERP_FNAME}"
            hyperp_info = load_json(hyperp_path)

            model_path = None
            if 'model' not in hyperp_info:
                model_path = f"{BASE_MODELS_DIR}/Undi95/Meta-Llama-3-8B-Instruct-hf"
            else:
                model_path = hyperp_info['model']
                model_path = f"{BASE_MODELS_DIR}/{'/'.join(model_path.split("/")[-2:])}"

            tokenizer = AutoTokenizer.from_pretrained(model_path)
            for i in tqdm(range(len(generation_info))):
                gen_answer = generation_info[i]['gen_answer']
                answer_tokens = tokenizer.encode(gen_answer)
                generation_info[i]['metainfo'][NUM_TOKENS_FIELD] = len(answer_tokens)

            save_json(geninfo_path, generation_info)
            

mtssquad logs_v2
mtssquad/logs_v2/v1.1.2


100%|██████████| 2000/2000 [00:00<00:00, 21518.30it/s]


mtssquad/logs_v2/v2.1.1


100%|██████████| 2000/2000 [00:00<00:00, 35648.12it/s]


mtssquad/logs_v2/v3


...

100%|██████████| 150/150 [00:00<00:00, 34742.15it/s]


mtssquad/logs_v2/v2.1.2


100%|██████████| 2000/2000 [00:00<00:00, 23139.33it/s]


mtssquad/logs_v2/v3.1.2


100%|██████████| 2000/2000 [00:00<00:00, 21744.54it/s]
